### **OpenCLIP con Docker, una GPU y plantillas SLURM**

Este cuaderno es el **centro didáctico** del proyecto, no toda la lógica del proyecto.

Su función es:

- explicar la lógica experimental,
- inspeccionar el subconjunto real ya incluido,
- correr el baseline preentrenado,
- revisar retrieval y hard negatives,
- mostrar cómo se escala a `torchrun` y SLURM,
- y dejar el pipeline automatizado en `scripts/` y `src/`.

#### **Idea principal**
El flujo realista de trabajo es:

1. trabajar primero con un checkpoint preentrenado,
2. extraer embeddings de un subconjunto real,
3. evaluar recuperación cruzada,
4. inspeccionar errores,
5. dejar el fine-tuning CSV como extensión controlada.


#### **Organización del proyecto**

```text
Proyecto/
├── Cuaderno9-MCC225.ipynb
├── README.md
├── requirements-extra.txt
├── configs/
├── data/
│   ├── bootstrap_flickr30k/
│   ├── raw/
│   ├── interim/
│   └── processed/
├── outputs/
├── reports/
├── scripts/
├── slurm/
└── src/
```

Este cuaderno asume que se ejecuta desde la carpeta `Proyecto/`.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Ejecuta este cuaderno desde la carpeta Semana4/")

sys.path.append(str(PROJECT_ROOT))

from src.io_utils import load_yaml
from src.dataset_utils import load_metadata
from src.openclip_utils import create_model, encode_image_paths, encode_texts
from src.metrics import summarize_ranking
from src.retrieval import topk_text_to_image, topk_image_to_text, mine_hard_negatives
from src.visualize import show_gallery, show_retrieval_results


#### **1. Cargar configuración local**


In [ ]:
cfg = load_yaml(PROJECT_ROOT / "configs/local.yaml")
cfg


#### **2. Verificación rápida del entorno**

Este cuaderno está alineado con el flujo Docker del curso.  
Antes de correr experimentos, conviene verificar:

- versión de PyTorch,
- visibilidad de CUDA,
- nombre de GPU,
- disponibilidad de `open_clip`.


In [ ]:
!python scripts/00_verify_env.py


#### **3. Inspección del subconjunto real ya incluido**

Aquí usamos un **bootstrap sample real de Flickr30k** ya materializado en el proyecto.

No es un dataset inventado: las imágenes provienen del dataset Flickr30k y sirven para:
- ver ejemplos reales,
- probar retrieval,
- estudiar hard negatives,
- demostrar la tubería sin depender de una descarga grande inicial.


In [ ]:
metadata = load_metadata(PROJECT_ROOT / "data/bootstrap_flickr30k/metadata.csv", root=PROJECT_ROOT)
metadata[["image_id", "label", "caption"]]


In [ ]:
fig = show_gallery(metadata, ncols=3, figsize=(12, 8))
plt.show()


#### **4. Cargar OpenCLIP**

Usaremos un baseline sólido y razonable para una sola GPU:

- `ViT-B-32`
- `laion2b_s34b_b79k`

Es un punto de partida realista para Semana 4: contraste, alineamiento y retrieval.


In [ ]:
model_name = cfg["model"]["model_name"]
pretrained = cfg["model"]["pretrained"]

model, preprocess, tokenizer, device = create_model(model_name, pretrained)
print("device =", device)
print("model_name =", model_name)
print("pretrained =", pretrained)


#### **5. Extraer embeddings del subconjunto bootstrap**


In [ ]:
image_features = encode_image_paths(
    model,
    preprocess,
    metadata["filepath"].tolist(),
    device=device,
    batch_size=cfg["runtime"]["batch_size"],
)

text_features = encode_texts(
    model,
    tokenizer,
    metadata["caption"].tolist(),
    device=device,
    batch_size=max(cfg["runtime"]["batch_size"], 32),
)

image_features.shape, text_features.shape


#### **6. Matriz de similitud y métricas de retrieval**

Como el bootstrap tiene pares alineados `(imagen_i, caption_i)`, la diagonal representa el match correcto.


In [ ]:
sim = image_features @ text_features.T
metrics_i2t = summarize_ranking(sim)
metrics_t2i = summarize_ranking(sim.T)

pd.DataFrame([
    {"direction": "image_to_text", **{k: v for k, v in metrics_i2t.items() if k != "Ranks"}},
    {"direction": "text_to_image", **{k: v for k, v in metrics_t2i.items() if k != "Ranks"}},
])


#### **7. Ejemplos de recuperación cruzada**


In [ ]:
query = "two men cooking in a kitchen"
query_feature = encode_texts(model, tokenizer, [query], device=device)
results = topk_text_to_image(query_feature, image_features, metadata, k=4)
results


In [ ]:
fig = show_retrieval_results(results, figsize=(12, 4))
plt.show()


In [ ]:
img_idx = 3
image_result = topk_image_to_text(image_features[img_idx:img_idx+1], text_features, metadata, k=4)
image_result


#### **8. Negativos duros**

Esta parte importa tanto como la métrica agregada. Los negativos duros muestran pares incorrectos con score alto y permiten discutir:

- correlaciones espurias,
- ambigüedad semántica,
- similitud superficial,
- límites del embedding compartido.


In [ ]:
hard = mine_hard_negatives(sim, metadata, top_n=8)
hard[["image_id", "image_label", "text_label", "score", "negative_caption"]]


In [ ]:
row = hard.iloc[0]
img = Image.open(row["image_filepath"]).convert("RGB")
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f'Imagen: {row["image_label"]}\nNegative caption score={row["score"]:.3f}')
plt.axis("off")
plt.show()

print("Caption correcto:")
print(row["image_caption"])
print()
print("Hard negative:")
print(row["negative_caption"])


#### **9. Zero-shot como extensión**

El proyecto trae una extensión zero-shot mínima.
En este bootstrap la columna `label` es pequeña y curada solo para demostración.


In [ ]:
!python scripts/02_build_embeddings.py   --metadata-csv data/bootstrap_flickr30k/metadata.csv   --model-name ViT-B-32   --pretrained laion2b_s34b_b79k   --batch-size 16   --output outputs/embeddings/bootstrap_embeddings.npz


In [ ]:
!python scripts/04_eval_zeroshot.py   --embeddings outputs/embeddings/bootstrap_embeddings.npz   --metadata-csv data/bootstrap_flickr30k/metadata.csv   --prompt-config data/bootstrap_flickr30k/prompt_config.json   --output-csv outputs/metrics/zeroshot_predictions.csv


In [ ]:
pd.read_csv(PROJECT_ROOT / "outputs/metrics/zeroshot_predictions.csv")


#### **10. Pipeline reproducible fuera del notebook**

La idea correcta no es dejar todo dentro del notebook.

El proyecto ya trae pipeline reproducible:

- `scripts/run_local_pipeline.sh`
- `scripts/10_train_openclip_csv_local.sh`
- `scripts/11_train_openclip_csv_torchrun.sh`
- `slurm/train_openclip_csv_single_node.sbatch`
- `slurm/train_openclip_csv_multi_node.sbatch`

**Pipeline local completo**
```bash
bash scripts/run_local_pipeline.sh
```

**Fine-tuning CSV local muy corto**
```bash
bash scripts/10_train_openclip_csv_local.sh
```

**Template torchrun**
```bash
bash scripts/11_train_openclip_csv_torchrun.sh
```


#### **11. Descargar un subconjunto mayor y más realista de Flickr30k**

Cuando quieras salir del bootstrap y pasar a algo más serio, ejecuta:


In [ ]:
# Descomenta cuando quieras materializar un subconjunto mayor
# !python scripts/01_prepare_flickr30k_from_hf.py #   --output-root data/processed/flickr30k_hf #   --train-limit 512 #   --val-limit 128 #   --test-limit 128


In [ ]:
!python scripts/01_prepare_flickr30k_from_hf.py \
  --output-root data/processed/flickr30k_hf \
  --train-limit 512 \
  --val-limit 128 \
  --test-limit 128

Después puedes cambiar el CSV de entrada en `scripts/02_build_embeddings.py` y repetir la evaluación.

Eso te deja una progresión clara:

- bootstrap real incluido,
- subconjunto mayor descargado desde HF,
- evaluación reproducible,
- extensión a entrenamiento CSV,
- y plantillas listas para SLURM.


#### **13. Entregables sugeridos**

1. Comparar dos checkpoints de OpenCLIP.
2. Ejecutar retrieval sobre el bootstrap y sobre un subconjunto más grande.
3. Reportar R@1, R@5, MRR y 5 hard negatives comentados.
4. Hacer una mini ablación:
   - sin captions alternas,
   - con distinto batch size,
   - o con distinto prompt set.
5. Escribir conclusiones en `reports/week4_findings.md`.
